In [2]:
# 2.1 Loading Necessary Libraries
import numpy as np  # linear algebra
import pandas as pd  # data processing
import matplotlib.pyplot as plt  # For basic data visualization.
import seaborn as sns  # For statistical data visualization
import warnings  # To manage warnings

# Suppressing FutureWarning
warnings.filterwarnings("ignore")

# 2.2 Loading Dataets
train_df = pd.read_csv("../../data/spaceship-titanic/train.csv")
test_df = pd.read_csv("../../data/spaceship-titanic/test.csv")

# Let's quickly check the data by viewing the first few rows.
print(train_df.head())
print(test_df.head())

  PassengerId HomePlanet CryoSleep  Cabin  Destination   Age    VIP  \
0     0001_01     Europa     False  B/0/P  TRAPPIST-1e  39.0  False   
1     0002_01      Earth     False  F/0/S  TRAPPIST-1e  24.0  False   
2     0003_01     Europa     False  A/0/S  TRAPPIST-1e  58.0   True   
3     0003_02     Europa     False  A/0/S  TRAPPIST-1e  33.0  False   
4     0004_01      Earth     False  F/1/S  TRAPPIST-1e  16.0  False   

   RoomService  FoodCourt  ShoppingMall     Spa  VRDeck               Name  \
0          0.0        0.0           0.0     0.0     0.0    Maham Ofracculy   
1        109.0        9.0          25.0   549.0    44.0       Juanna Vines   
2         43.0     3576.0           0.0  6715.0    49.0      Altark Susent   
3          0.0     1283.0         371.0  3329.0   193.0       Solam Susent   
4        303.0       70.0         151.0   565.0     2.0  Willy Santantines   

   Transported  
0        False  
1         True  
2        False  
3        False  
4         True  
  

In [3]:
y_train = train_df["Transported"]
train_len = len(train_df)

X_train_temp = train_df.drop(columns=["Transported"])

all_data = pd.concat([X_train_temp, test_df], axis=0).reset_index(drop=True)


def missing_data_table(df):
    total = (
        df.isnull().sum().sort_values(ascending=False)
    )  # Total missing values in each column.
    percent = (df.isnull().sum() / df.isnull().count()).sort_values(
        ascending=False
    )  # Percentage of missing values.

    missing_data = pd.concat(
        [total, percent], axis=1, keys=["Total", "Percent"]
    )  # Combine the results.

    # Filter columns that actually have missing values.
    return missing_data[missing_data["Total"] > 0]


# Display missing data for the training dataset.
missing_data = missing_data_table(all_data)
missing_data

,Total,Percent
CryoSleep,310,0.023901
ShoppingMall,306,0.023593
Cabin,299,0.023053
VIP,296,0.022822
Name,294,0.022668
FoodCourt,289,0.022282
HomePlanet,288,0.022205
Spa,284,0.021897
Destination,274,0.021126
Age,270,0.020817


In [4]:
all_data["CryoSleep"] = all_data["CryoSleep"].fillna("Unknown")
all_data["HomePlanet"] = all_data["HomePlanet"].fillna("Unknown")
all_data["Destination"] = all_data["Destination"].fillna("Unknown")

all_data["ShoppingMall"] = all_data["ShoppingMall"].fillna(
    all_data["ShoppingMall"].median()
)
all_data["FoodCourt"] = all_data["FoodCourt"].fillna(all_data["FoodCourt"].median())
all_data["Spa"] = all_data["Spa"].fillna(all_data["Spa"].median())
all_data["VRDeck"] = all_data["VRDeck"].fillna(all_data["VRDeck"].median())
all_data["RoomService"] = all_data["RoomService"].fillna(
    all_data["RoomService"].median()
)

all_data["total_spending"] = all_data[
    ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
].sum(axis=1)

all_data["Age"] = all_data["Age"].fillna(all_data["Age"].median())

all_data["CabinDeck"] = all_data["Cabin"].str.split("/").str[0]
all_data["CabinNum"] = all_data["Cabin"].str.split("/").str[1]
all_data["CabinSide"] = all_data["Cabin"].str.split("/").str[2]

all_data["CabinDeck"] = all_data["CabinDeck"].fillna("Unknown")
all_data["CabinNum"] = all_data["CabinNum"].fillna("Unknown")
all_data["CabinSide"] = all_data["CabinSide"].fillna("Unknown")

all_data["VIP"] = all_data["VIP"].apply(lambda x: 1 if x is True else 0)

all_data["last_name"] = all_data["Name"].str.split(" ").str[1]

all_data["last_name"] = all_data["last_name"].fillna("Unknown")

all_data = all_data.drop(columns=["Cabin", "Name"])

In [5]:
missing_data = missing_data_table(all_data)
missing_data

,Total,Percent


In [6]:
all_data["PassengerGroup"] = all_data["PassengerId"].str.split("_").str[0]

all_data["PassengerGroup_size"] = all_data.groupby("PassengerGroup")[
    "PassengerGroup"
].transform("count")


def categorize_group(size):
    if size == 1:
        return "Solo"
    elif size <= 4:
        return "Small_Group"
    else:
        return "Large_Group"


all_data["PassengerGroup_type"] = all_data["PassengerGroup_size"].apply(
    categorize_group
)

all_data["family_size"] = all_data.groupby(["PassengerGroup", "last_name"])[
    "PassengerId"
].transform("count")

In [7]:
missing_data = missing_data_table(all_data)
missing_data

,Total,Percent


In [14]:
all_data = all_data.drop(
    columns=[
        "PassengerId",
        "RoomService",
        "FoodCourt",
        "ShoppingMall",
        "Spa",
        "VRDeck",
        "PassengerGroup_size",
    ],
    errors="ignore",
)

X_train = all_data.iloc[:train_len, :]
X_test = all_data.iloc[train_len:, :]

In [15]:
cat_feature_indices = [
    "HomePlanet",
    "CryoSleep",
    "Destination",
    "VIP",
    "CabinDeck",
    "CabinNum",
    "CabinSide",
    "last_name",
    "PassengerGroup",
    "PassengerGroup_type",
]

# Convert columns in place
for col in cat_feature_indices:
    X_train[col] = X_train[col].astype(str).astype("category")
    # Do the same for your test/validation sets if they are separate
    X_test[col] = X_test[col].astype(str).astype("category")

In [23]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000/")
mlflow.set_experiment("XGBoost_Optuna_SpaceTitanic")

2026/01/09 15:16:27 INFO mlflow.tracking.fluent: Experiment with name 'XGBoost_Optuna_SpaceTitanic' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/4', creation_time=1767946587106, experiment_id='4', last_update_time=1767946587106, lifecycle_stage='active', name='XGBoost_Optuna_SpaceTitanic', tags={}>

In [ ]:
import optuna
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import log_loss


def objective(trial):
    # 1. Define the search space
    param = {
        "verbosity": 0,
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
        "device": "cuda",  # Modern XGBoost GPU syntax
        "lambda": trial.suggest_float("lambda", 1e-8, 10.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-8, 10.0, log=True),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "max_depth": trial.suggest_int("max_depth", 3, 9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "subsample": trial.suggest_float("subsample", 0.4, 1.0),
        "n_estimators": 2000,
        "early_stopping_rounds": 50,
        "enable_categorical": True,  # Required if X_train has 'category' dtypes
    }

    # 2. Setup K-Fold
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = []

    # Start a nested MLflow run for each trial
    with mlflow.start_run(nested=True):
        for train_idx, val_idx in kf.split(X_train, y_train):
            X_t, X_v = X_train.iloc[train_idx], X_train.iloc[val_idx]
            y_t, y_v = y_train.iloc[train_idx], y_train.iloc[val_idx]

            model = xgb.XGBClassifier(**param)
            model.fit(X_t, y_t, eval_set=[(X_v, y_v)], verbose=False)

            preds = model.predict_proba(X_v)
            score = log_loss(y_v, preds)
            cv_scores.append(score)

        avg_logloss = np.mean(cv_scores)

        # Log trial results to MLflow
        mlflow.log_params(trial.params)
        mlflow.log_metric("avg_logloss", avg_logloss)

        return avg_logloss

with mlflow.start_run(run_name="Main_Optimization"):
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=30)  # Total trials

    # Log best params
    mlflow.log_params(study.best_params)
    mlflow.log_metric("best_logloss", study.best_value)

In [27]:
best_params = {
    "verbosity": 0,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "device": "cuda",  # Modern XGBoost GPU syntax
    "lambda": 1.154541981560131,
    "alpha": 8.010507327897328,
    "learning_rate": 0.030312266341625486,
    "max_depth": 5,
    "colsample_bytree": 0.9273754418897571,
    "subsample": 0.9155300109615088,
    "n_estimators": 2000,
    # "early_stopping_rounds": 50,
    "enable_categorical": True,  # Required if X_train has 'category' dtypes
}

model = xgb.XGBClassifier(**best_params)

model.fit(X_train, y_train, verbose=False)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.9273754418897571
,device,'cuda'
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,'logloss'


In [29]:
id_col = test_df["PassengerId"]


def export_submission(model, id_col):
    preds = model.predict(X_test)
    preds_bool = preds.astype(bool)
    submission = pd.DataFrame({"PassengerId": id_col, "Transported": preds_bool})

    model_name = type(model).__name__
    submission.to_csv(f"{model_name}_submission.csv", index=False)


export_submission(model, id_col)